In [1]:
import re
import pandas as pd

df = pd.read_csv("../data/processed/netflix_clean.csv", parse_dates=["date_added"])

def to_tokens(value):
    if value == "Unknown":
        return ""
    parts = [re.sub(r"[^\w]", "", p) for p in value.split(",")]
    return " ".join(parts)

df["features"] = (
    df["type"].apply(to_tokens) + " "
    + df["rating"].apply(to_tokens) + " "
    + df["listed_in"].apply(to_tokens) + " "
    + df["director"].apply(to_tokens) + " "
    + df["country"].apply(to_tokens)
).str.lower().str.split().str.join(" ")

In [2]:
print(df.shape)
print(df.loc[[0, 1, 2], ["title", "features"]].to_string())
print((df["features"].str.len() == 0).sum(), df["features"].str.split().str.len().describe()[["min", "mean", "max"]])

(8786, 18)
                  title                                                                               features
0  Dick Johnson Is Dead                                   movie pg13 documentaries kirstenjohnson unitedstates
1             Ganglands  tvshow tvma crimetvshows internationaltvshows tvactionadventure julienleclercq france
2         Midnight Mass                    tvshow tvma tvdramas tvhorror tvmysteries mikeflanagan unitedstates
0 min      4.000000
mean     5.961074
max     18.000000
Name: features, dtype: float64


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

tfidf = TfidfVectorizer(token_pattern=r"\S+")
matrix = tfidf.fit_transform(df["features"])
print(matrix.shape, len(tfidf.vocabulary_))

(8786, 5127) 5127


In [4]:
def recommend(title, n=5):
    matches = df.index[df["title"].str.lower() == title.lower()]
    if len(matches) == 0:
        return "Title not found"
    i = matches[0]
    scores = linear_kernel(matrix[i], matrix).ravel()
    picks = []
    for j in scores.argsort()[::-1]:
        if df.loc[j, "title"].lower() == title.lower():
            continue
        picks.append(j)
        if len(picks) == n:
            break
    result = df.loc[picks, ["title", "type", "rating", "listed_in", "director"]].copy()
    result["similarity"] = scores[picks].round(3)
    return result

recommend("Dick Johnson Is Dead")

,title,type,rating,listed_in,director,similarity
7436,The Irishman: In Conversation,Movie,TV-MA,Documentaries,Unknown,0.336
8045,Liberated: The New Sexual Revolution,Movie,TV-MA,Documentaries,Unknown,0.336
7264,Murder to Mercy: The Cyntoia Brown Story,Movie,TV-MA,Documentaries,Unknown,0.336
8309,Virunga: Gorillas in Peril,Movie,TV-14,Documentaries,Unknown,0.323
7288,Surviving R. Kelly: The Impact,Movie,TV-14,Documentaries,Unknown,0.323


In [5]:
selected = ["Ganglands", "Midnight Mass", "Motu Patlu in Wonderland", "Sankofa"]
for t in selected:
    print("Recommendations for:", t)
    display(df[df["title"] == t][["title", "type", "rating", "listed_in", "director"]])
    display(recommend(t))

Recommendations for: Ganglands


,title,type,rating,listed_in,director
1,Ganglands,TV Show,TV-MA,"Crime TV Shows, International TV Shows, TV Act...",Julien Leclercq


,title,type,rating,listed_in,director,similarity
1182,Sentinelle,Movie,TV-MA,"Action & Adventure, Dramas, International Movies",Julien Leclercq,0.733
2141,Earth and Blood,Movie,TV-MA,"Dramas, International Movies, Thrillers",Julien Leclercq,0.725
6734,Lupin,TV Show,TV-MA,"Crime TV Shows, International TV Shows, TV Act...",Unknown,0.716
8404,Crime Time,TV Show,TV-MA,"Crime TV Shows, International TV Shows, TV Act...",Unknown,0.716
7032,La Révolution,TV Show,TV-MA,"International TV Shows, TV Action & Adventure,...",Unknown,0.590


Recommendations for: Midnight Mass


,title,type,rating,listed_in,director
2,Midnight Mass,TV Show,TV-MA,"TV Dramas, TV Horror, TV Mysteries",Mike Flanagan


...

,title,type,rating,listed_in,director
9,Motu Patlu in Wonderland,Movie,TV-Y7,"Children & Family Movies, Music & Musicals",Suhas Kadav


,title,type,rating,listed_in,director,similarity
1063,Motu Patlu in Hong Kong: Kung Fu Kings 3,Movie,TV-Y7,"Children & Family Movies, Music & Musicals",Suhas Kadav,1.00
1066,Motu Patlu Kung Fu Kings 4 The Challenge of Ku...,Movie,TV-Y7,"Children & Family Movies, Music & Musicals",Suhas Kadav,1.00
7,Motu Patlu in the Game of Zones,Movie,TV-Y7,"Children & Family Movies, Comedies, Music & Mu...",Suhas Kadav,0.97
1065,Motu Patlu in the City of Gold,Movie,TV-Y7,"Children & Family Movies, Comedies, Music & Mu...",Suhas Kadav,0.97
1068,Motu Patlu the Superheroes – Super Villains fr...,Movie,TV-Y7,Children & Family Movies,Suhas Kadav,0.92


Recommendations for: Sankofa


,title,type,rating,listed_in,director
4,Sankofa,Movie,TV-MA,"Dramas, Independent Movies, International Movies",Haile Gerima


,title,type,rating,listed_in,director,similarity
8703,The Darkest Dawn,Movie,TV-MA,"Action & Adventure, Independent Movies, Intern...",Unknown,0.301
7818,Black Mirror: Bandersnatch,Movie,TV-MA,"Dramas, International Movies, Sci-Fi & Fantasy",Unknown,0.283
4735,Dagh Ujala,Movie,TV-MA,"Dramas, Independent Movies, International Movies",Raj Amit Kumar,0.265
7803,Adhugo,Movie,TV-14,"Comedies, Independent Movies, International Mo...",Unknown,0.263
7798,Polar,Movie,TV-MA,"Action & Adventure, International Movies",Unknown,0.263


In [6]:
import numpy as np

genre_sets = df["listed_in"].str.split(",").apply(lambda g: {x.strip() for x in g})
rng = np.random.default_rng(42)
sample = rng.choice(len(df), size=300, replace=False)

def top_n(i, n=5):
    scores = linear_kernel(matrix[i], matrix).ravel()
    out = []
    for j in scores.argsort()[::-1]:
        if df.at[j, "title"].lower() == df.at[i, "title"].lower():
            continue
        out.append(j)
        if len(out) == n:
            break
    return out

def rates(i, js):
    same_type = np.mean([df.at[j, "type"] == df.at[i, "type"] for j in js])
    shares_genre = np.mean([len(genre_sets[i] & genre_sets[j]) > 0 for j in js])
    return same_type, shares_genre

In [7]:
model = np.array([rates(i, top_n(i)) for i in sample])
random_picks = np.array([rates(i, rng.choice(len(df), size=5, replace=False)) for i in sample])
print("model  (same type, shares genre):", model.mean(axis=0).round(3))
print("random (same type, shares genre):", random_picks.mean(axis=0).round(3))

model  (same type, shares genre): [0.933 0.899]
random (same type, shares genre): [0.595 0.241]
